# **Aggregates, GROUP BY, and HAVING**

This lesson uses one small library-loans table throughout. Each row represents one loan. We’ll use it to learn how SQL summarizes data and answers questions such as “How many loans does each member have?” and “How much fine does each member owe?”

In [ ]:
%pip install --force-reinstall jupysql

In [1]:
%load_ext sql

In [ ]:
%%sql sqlite://

CREATE TABLE loans (
    loan_id INTEGER PRIMARY KEY,
    member_id INTEGER NOT NULL,
    book_title TEXT NOT NULL,
    loan_days INTEGER NOT NULL CHECK (loan_days >= 0),
    fine_cents INTEGER NOT NULL CHECK (fine_cents >= 0)
);

INSERT INTO loans (loan_id, member_id, book_title, loan_days, fine_cents)
VALUES
    (101, 1, 'Python Basics', 7, 150),
    (102, 1, 'SQL for Beginners', 5, 0),
    (103, 2, 'Web APIs', 10, 300),
    (104, 2, 'Data Structures', 6, 100),
    (105, 3, 'PostgreSQL Guide', 8, 0);

This is how the table will look like:
## loans

| loan_id | member_id | book_title | loan_days | fine_cents |
|---:|---:|---|---:|---:|
| 101 | 1 | Python Basics | 7 | 150 |
| 102 | 1 | SQL for Beginners | 5 | 0 |
| 103 | 2 | Web APIs | 10 | 300 |
| 104 | 2 | Data Structures | 6 | 100 |
| 105 | 3 | PostgreSQL Guide | 8 | 0 |


## 1. **Aggregate functions:**

An **aggregate function** combines values from several rows and returns a summary. PostgreSQL includes `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX`.

### COUNT(*): count rows

This counts how many loan records are in the table.

In [4]:
%config SqlMagic.style = 'DEFAULT'
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [ ]:
%%sql

SELECT COUNT(*) AS loan_count
FROM loans;

`COUNT(*)` counts rows, even when some columns in a row contain `NULL`.

### SUM(column): add values

This adds all the fine amounts.

In [ ]:
%%sql
SELECT SUM(fine_cents) AS total_fines_cents
FROM loans;

### AVG(column): calculate the average

This finds the average number of days books were loaned.

In [ ]:
%%sql
SELECT AVG(loan_days) AS average_loan_days
FROM loans;

The calculation is `(7 + 5 + 10 + 6 + 8) / 5`, which equals `7.2`.

### MIN(column): find the smallest value

This finds the shortest loan period.

In [ ]:
%%sql
SELECT MIN(loan_days) AS shortest_loan_days
FROM loans;

### MAX(column): find the largest value

This finds the longest loan period.

In [ ]:
%%sql
SELECT MAX(loan_days) AS longest_loan_days
FROM loans;

### A note about NULL

`NULL` means a value is missing or unknown. Most aggregate functions ignore `NULL` values. For example, `COUNT(fine_cents)` counts rows where `fine_cents` has a value; `COUNT(*)` counts all rows.

If `SUM` has no values to add, it can return `NULL`. Use `COALESCE` to show zero instead:

In [ ]:
%%sql
SELECT COALESCE(SUM(fine_cents), 0) AS total_fines_cents
FROM loans;

`COALESCE(value, 0)` means “use the value if it exists; otherwise use zero.”

## 2. GROUP BY: make one summary per group

An aggregate without `GROUP BY` summarizes the whole table. `GROUP BY` divides rows into groups with matching values, then calculates a summary for each group.

This query counts loans per member:

In [ ]:
%%sql

SELECT
    member_id,
    COUNT(*) AS loan_count
FROM loans
GROUP BY member_id
ORDER BY member_id;

SQL puts member 1’s two loan rows together, member 2’s two rows together, and member 3’s one row together. Then it counts each group.

`ORDER BY member_id` sorts the results. Without `ORDER BY`, SQL does not promise a particular order.

### A rule for selected columns

In a grouped query, every selected item must either:

- Appear in the `GROUP BY`, or
- Be used inside an aggregate function.

This works because `member_id` is grouped and `COUNT(*)` is an aggregate:

In [ ]:
%%sql

SELECT member_id, COUNT(*) AS loan_count
FROM loans
GROUP BY member_id;

This is unclear because a member can have multiple book titles:

In [ ]:
%%sql

SELECT member_id, book_title, COUNT(*)
FROM loans
GROUP BY member_id;

SQL would not know which book title to show for each member. To show a count per book as well, group by both columns:

In [ ]:
%%sql
SELECT
    member_id,
    book_title,
    COUNT(*) AS loan_count
FROM loans
GROUP BY member_id, book_title
ORDER BY member_id, book_title;

## 3. WHERE: filter individual rows

`WHERE` selects which rows are included **before** grouping or calculating aggregates.

For example, show only loans that have a fine:

In [ ]:
%%sql
SELECT loan_id, member_id, fine_cents
FROM loans
WHERE fine_cents > 0
ORDER BY loan_id;

`WHERE fine_cents > 0` checks each loan row separately and leaves out the rows with zero fines.

Use `WHERE` for conditions on individual rows, such as a particular member, book, date, or fine amount.

## 4. HAVING: filter groups

`HAVING` keeps or removes groups **after** SQL has formed the groups and calculated their aggregate values.

This query shows members whose total fine is at least 200 cents:

In [ ]:
%%sql

SELECT
    member_id,
    SUM(fine_cents) AS total_fine_cents
FROM loans
GROUP BY member_id
HAVING SUM(fine_cents) >= 200
ORDER BY member_id;

Member 1 owes 150 cents, member 2 owes 400 cents, and member 3 owes zero. Only member 2 meets the `HAVING` condition.

### The difference between WHERE and HAVING

- `WHERE` filters **individual rows** before grouping.
- `HAVING` filters **groups** after calculating aggregates.

This is invalid:

In [ ]:
%%sql

SELECT member_id, SUM(fine_cents)
FROM loans
WHERE SUM(fine_cents) >= 200
GROUP BY member_id;

At the point where `WHERE` runs, SQL has not calculated each member’s sum yet. Use `HAVING` for this condition.

## Query guide

| If you want to… | Use… |
|---|---|
| Count rows or summarize values | An aggregate such as `COUNT`, `SUM`, `AVG`, `MIN`, or `MAX` |
| Calculate a summary for each member | `GROUP BY member_id` |
| Remove individual loan rows before grouping | `WHERE` |
| Remove member groups based on their totals | `HAVING` |
| Sort the output | `ORDER BY` |

A typical clause order is:

```sql
SELECT ...
FROM ...
WHERE ...
GROUP BY ...
HAVING ...
ORDER BY ...;
```

Remember the key idea: **`WHERE` filters loan rows; `HAVING` filters grouped member results.** Your fine report groups loans by member, adds the fines in each group, and uses integer cents or kobo for exact money calculations.